# Product Catalog and Seller Scope

This notebook validates the complete Amazon Beauty product catalog, exact review-to-metadata coverage, a candidate conditioner niche, and the mechanics of an explicit seller-versus-competitor workspace.

## Objectives

- validate every registered catalog record and its metadata-quality flags;
- measure exact join coverage for the complete canonical review artifact;
- resolve a niche through a stable Amazon category-path ID;
- inspect products and review volume before approving the niche;
- demonstrate explicit, disjoint seller and competitor product sets.

# Каталог товаров и область анализа продавца

Этот ноутбук проверяет полный каталог Beauty-товаров Amazon, точное покрытие соединения отзывов с метаданными, кандидатную нишу кондиционеров и механику явной рабочей области «товар продавца против конкурентов».

## Цели

- проверить каждую зарегистрированную запись каталога и ее флаги качества;
- измерить точное покрытие соединения полного канонического набора отзывов;
- определить нишу через стабильный ID пути Amazon-категории;
- посмотреть товары и объем отзывов до утверждения ниши;
- показать явные непересекающиеся наборы товаров продавца и конкурентов.

## Inputs and outputs

Inputs are the registered full product catalog, canonical reviews, category registry, and their exact quality reports. Heavy source parsing and joins were performed by `src.analytics.product_catalog`; this notebook queries the versioned outputs with DuckDB.

The seller/competitor example is deliberately marked as a demonstration and is not persisted. The public dataset does not reveal which ASINs belong to a future user, so a real workspace must be created from explicit user input.

## Входы и результаты

На вход подаются зарегистрированный полный каталог, канонические отзывы, реестр категорий и точные отчеты качества. Тяжелое чтение исходника и соединения выполнил модуль `src.analytics.product_catalog`; ноутбук запрашивает версионные результаты через DuckDB.

Пример продавца и конкурентов специально отмечен как демонстрационный и не сохраняется. Публичный датасет не сообщает, какие ASIN принадлежат будущему пользователю, поэтому реальная рабочая область должна создаваться только из явного выбора пользователя.

In [ ]:
# Standard library / Стандартная библиотека
import json
import sys
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import duckdb
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Locate the repository root before importing local project modules.
# Находим корень репозитория до импорта локальных модулей проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.analytics.product_catalog import PRODUCT_CATALOG_SCHEMA
from src.common.project import find_project_root
from src.ingestion.dataset_manifest import (
    load_dataset_manifest,
    manifest_is_valid,
    verify_dataset_manifest,
)
from src.schemas.workspace import (
    ProductNicheDefinition,
    SellerWorkspaceScope,
)

In [ ]:
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "config/datasets/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
CATALOG_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/data_quality/amazon_reviews_2023_beauty_2021_2023_v1_product_catalog.json"
)
STANDARD_CONDITIONERS_PATH = (
    "Beauty & Personal Care > Hair Care > Shampoo & Conditioner > Conditioners"
)
DEMO_COMPETITOR_COUNT = 3

manifest = load_dataset_manifest(MANIFEST_PATH)
catalog_registration = manifest.file_by_role("product_catalog")
PRODUCT_CATALOG_PATH = PROJECT_ROOT / catalog_registration.path
CANONICAL_REVIEWS_PATH = (
    PROJECT_ROOT / manifest.file_by_role("canonical_reviews").path
)
CATEGORY_REGISTRY_PATH = (
    PROJECT_ROOT / manifest.file_by_role("category_registry").path
)

print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Catalog schema: {manifest.schema_versions.product_catalog}")

## 1. Complete catalog contract

The catalog contract requires exactly one row per `parent_asin`; the builder rejects duplicate or missing metadata keys before any review join can multiply rows. It preserves normalized structured category paths, features and descriptions, stores source details as JSON, and adds exact review-coverage columns. Missing optional metadata does not remove a product; `metadata_quality_flags` explains the limitation.

## Контракт полного каталога

Контракт каталога требует ровно одну строку на каждый `parent_asin`: построитель отклоняет отсутствующие и повторяющиеся ключи метаданных до соединения, поэтому отзывы не могут размножиться. Каталог сохраняет нормализованные структурированные пути категорий, features и description, хранит исходные details в JSON и добавляет точные показатели покрытия отзывами. Отсутствие необязательных метаданных не удаляет товар — ограничение объясняется через `metadata_quality_flags`.

In [ ]:
if not CATALOG_REPORT_PATH.is_file():
    raise FileNotFoundError(CATALOG_REPORT_PATH)
manifest_results = verify_dataset_manifest(
    manifest,
    project_root=PROJECT_ROOT,
    verify_checksums=False,
    verify_record_counts=False,
)
if not manifest_is_valid(manifest_results):
    raise ValueError("One or more registered dataset artifacts are invalid")

catalog_report = json.loads(CATALOG_REPORT_PATH.read_text(encoding="utf-8"))
catalog_file = pq.ParquetFile(PRODUCT_CATALOG_PATH)
actual_schema = catalog_file.schema_arrow
schema_matches = actual_schema.equals(
    PRODUCT_CATALOG_SCHEMA, check_metadata=False
)
required_catalog_columns = [
    field.name for field in PRODUCT_CATALOG_SCHEMA if not field.nullable
]
null_expressions = ", ".join(
    f'count(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
    for column in required_catalog_columns
)
connection = duckdb.connect()
try:
    catalog_cardinality = connection.execute(
        """
        SELECT
            count(*) AS catalog_rows,
            count(DISTINCT parent_asin) AS distinct_parent_asins,
            count(*) - count(DISTINCT parent_asin) AS duplicate_parent_asins,
            sum(review_count) AS catalog_review_count
        FROM read_parquet(?)
        """,
        [str(PRODUCT_CATALOG_PATH)],
    ).fetchdf().iloc[0]
    required_null_values = connection.execute(
        f"SELECT {null_expressions} FROM read_parquet(?)",
        [str(PRODUCT_CATALOG_PATH)],
    ).fetchone()
finally:
    connection.close()
required_null_counts = dict(
    zip(required_catalog_columns, required_null_values, strict=True)
)

assert catalog_report["metadata_input_rows"] == catalog_report["output_rows"]
assert catalog_report["output_rows"] == catalog_report["distinct_parent_asin_count"]
assert catalog_file.metadata.num_rows == catalog_registration.record_count
assert catalog_file.metadata.num_rows == catalog_report["output_rows"]
assert schema_matches
assert catalog_cardinality["catalog_rows"] == catalog_report["output_rows"]
assert catalog_cardinality["distinct_parent_asins"] == catalog_report["distinct_parent_asin_count"]
assert catalog_cardinality["duplicate_parent_asins"] == 0
assert all(count == 0 for count in required_null_counts.values())
display(
    pd.Series(
        {
            "catalog_rows": catalog_report["output_rows"],
            "distinct_parent_asins": catalog_report["distinct_parent_asin_count"],
            "catalog_schema_passed": schema_matches,
            "required_nullability_passed": True,
            "duplicate_parent_asins": int(
                catalog_cardinality["duplicate_parent_asins"]
            ),
            "catalog_reconciliation_passed": True,
        },
        name="value",
    ).to_frame()
)
display(
    pd.DataFrame(
        {
            "column": PRODUCT_CATALOG_SCHEMA.names,
            "type": [str(field.type) for field in actual_schema],
            "expected_nullable": [
                field.nullable for field in PRODUCT_CATALOG_SCHEMA
            ],
            "physical_nullable": [field.nullable for field in actual_schema],
            "required_null_count": [
                required_null_counts.get(field.name)
                for field in PRODUCT_CATALOG_SCHEMA
            ],
        }
    )
)

## 2. Metadata quality

A missing price is not the same as an invalid price. Missing means Amazon supplied no value; invalid means a value existed but could not be safely interpreted as one exact number—for example `from 28.00`. The system stores null in both numeric cells but uses different flags, so later UI and analytics can explain the reason honestly.

## Качество метаданных

Отсутствующая цена и некорректная цена — разные случаи. Missing означает, что Amazon не передал значение; invalid — значение было, но его нельзя безопасно считать одним точным числом, например `from 28.00`. В числовой колонке оба случая становятся null, но получают разные флаги, чтобы интерфейс и аналитика честно объясняли причину.

In [ ]:
quality_summary = pd.DataFrame(
    {
        "quality_issue": [
            "missing product title",
            "missing main_category",
            "missing category path",
            "missing store",
            "missing price",
            "invalid price",
            "unmatched category path",
        ],
        "product_count": [
            catalog_report["missing_product_title_rows"],
            catalog_report["missing_main_category_rows"],
            catalog_report["missing_category_path_rows"],
            catalog_report["missing_store_rows"],
            catalog_report["missing_price_rows"],
            catalog_report["invalid_price_rows"],
            catalog_report["unmatched_category_path_rows"],
        ],
    }
)
quality_summary["product_share_pct"] = (
    quality_summary["product_count"] / catalog_report["output_rows"] * 100
)
display(quality_summary)

## 3. Review-to-product coverage

Review join coverage answers: 'Can every canonical review find its parent product metadata?' Product review coverage answers a different question: 'What share of catalog products has at least one review in our 2021–2023 analytical window?' A complete join can coexist with a much smaller reviewed-product share because the catalog includes products with no review in that period.

## Покрытие отзывов товарами

Покрытие соединения отзывов отвечает на вопрос: «Для каждого ли канонического отзыва найдены метаданные родительского товара?» Покрытие товаров отзывами отвечает на другой вопрос: «У какой доли каталога есть хотя бы один отзыв в окне 2021–2023?» Соединение может быть полным, а доля товаров с отзывами — значительно ниже, потому что каталог содержит товары без отзывов за выбранный период.

In [ ]:
assert catalog_report["canonical_review_rows"] == (
    catalog_report["matched_review_rows"]
    + catalog_report["unmatched_review_rows"]
)
assert int(catalog_cardinality["catalog_review_count"]) == catalog_report["matched_review_rows"]
display(
    pd.Series(
        {
            "canonical_reviews": catalog_report["canonical_review_rows"],
            "matched_reviews": catalog_report["matched_review_rows"],
            "unmatched_reviews": catalog_report["unmatched_review_rows"],
            "review_join_coverage_pct": catalog_report["review_join_coverage"] * 100,
            "catalog_products": catalog_report["output_rows"],
            "catalog_products_with_reviews": catalog_report["catalog_products_with_reviews"],
            "catalog_product_review_coverage_pct": catalog_report["catalog_product_review_coverage"] * 100,
        },
        name="value",
    ).to_frame()
)

## 4. Candidate standard-conditioner niche

For this first scope inspection we use exactly one leaf path: `Beauty & Personal Care > Hair Care > Shampoo & Conditioner > Conditioners`. This excludes the separate Amazon leaves for deep conditioners, 2-in-1 products, beard conditioners, color conditioners, and shampoos; it does not claim that every listing inside the selected leaf is the same product subtype. The status remains `candidate` until we review real products and business usefulness.

## Кандидатная ниша обычных кондиционеров

Для первой проверки области берем ровно один лист: `Beauty & Personal Care > Hair Care > Shampoo & Conditioner > Conditioners`. Отдельные Amazon-листья для глубоких кондиционеров, средств 2-в-1, продуктов для бороды, окрашенных волос и шампуней не входят в выбор; это не означает, что все листинги внутри выбранного пути относятся к одному подтипу товара. Статус остается `candidate` («кандидат»), пока мы не проверим реальные товары и полезность для бизнеса.

In [ ]:
connection = duckdb.connect()
try:
    niche_registry_row = connection.execute(
        """
        SELECT category_path_id, category_path_text, product_record_count
        FROM read_parquet(?)
        WHERE category_path_text = ?
        """,
        [str(CATEGORY_REGISTRY_PATH), STANDARD_CONDITIONERS_PATH],
    ).fetchdf()
finally:
    connection.close()

assert len(niche_registry_row) == 1
candidate_niche = ProductNicheDefinition(
    niche_id="hair_conditioners",
    niche_version="hair_conditioners_candidate_v1",
    dataset_version=manifest.dataset_version,
    category_registry_schema_version=manifest.schema_versions.category_registry,
    display_name="Standard Hair Conditioners",
    status="candidate",
    category_path_ids=[niche_registry_row.loc[0, "category_path_id"]],
)
display(pd.Series(candidate_niche.model_dump(), name="value").to_frame())

In [ ]:
connection = duckdb.connect()
try:
    niche_summary = connection.execute(
        """
        SELECT
            count(*) AS catalog_products,
            count(*) FILTER (WHERE review_count > 0) AS reviewed_products,
            sum(review_count) AS review_count,
            sum(reviewed_asin_count) AS reviewed_variation_asins
        FROM read_parquet(?)
        WHERE category_path_id IN (SELECT * FROM unnest(?))
        """,
        [str(PRODUCT_CATALOG_PATH), candidate_niche.category_path_ids],
    ).fetchdf()
    top_niche_products = connection.execute(
        """
        SELECT
            parent_asin,
            product_title,
            store,
            average_rating,
            rating_number,
            review_count,
            reviewed_asin_count
        FROM read_parquet(?)
        WHERE category_path_id IN (SELECT * FROM unnest(?))
            AND review_count > 0
        ORDER BY review_count DESC, parent_asin
        LIMIT 10
        """,
        [str(PRODUCT_CATALOG_PATH), candidate_niche.category_path_ids],
    ).fetchdf()
finally:
    connection.close()

display(niche_summary.T.rename(columns={0: "value"}))
display(top_niche_products)

## 5. Seller-versus-competitor workspace

The following scope uses the highest-review products only to demonstrate validation and querying. The first product is labelled `seller_demo` and the next three `competitors_demo`; this does not claim ownership, market leadership, or true competitive relationships. A real seller must explicitly provide or confirm these ASINs.

## Рабочая область продавца и конкурентов

Следующая область использует товары с наибольшим количеством отзывов только для демонстрации проверки и запросов. Первый товар условно называется `seller_demo`, следующие три — `competitors_demo`; это не означает владение, лидерство на рынке или реальные конкурентные отношения. Настоящий продавец должен явно указать или подтвердить ASIN.

In [ ]:
seller_demo_asins = [top_niche_products.loc[0, "parent_asin"]]
competitor_demo_asins = top_niche_products.loc[
    1:DEMO_COMPETITOR_COUNT, "parent_asin"
].tolist()
demo_scope = SellerWorkspaceScope(
    workspace_id="conditioner_demo",
    workspace_version="conditioner_demo_v1",
    dataset_version=manifest.dataset_version,
    niche=candidate_niche,
    seller_parent_asins=seller_demo_asins,
    competitor_parent_asins=competitor_demo_asins,
    review_date_start=manifest.date_window.start,
    review_date_end=manifest.date_window.end,
    verified_purchase_only=False,
)
display(pd.Series(demo_scope.model_dump(), name="value").to_frame())

In [ ]:
all_scope_asins = (
    demo_scope.seller_parent_asins + demo_scope.competitor_parent_asins
)
connection = duckdb.connect()
try:
    scope_review_summary = connection.execute(
        """
        SELECT
            CASE
                WHEN parent_asin IN (SELECT * FROM unnest(?))
                    THEN 'seller_demo'
                ELSE 'competitors_demo'
            END AS scope_group,
            count(*) AS review_count,
            count(DISTINCT parent_asin) AS product_count,
            avg(rating) AS mean_rating,
            count(*) FILTER (WHERE rating <= 2) AS low_rating_reviews,
            count(*) FILTER (WHERE verified_purchase) AS verified_reviews
        FROM read_parquet(?)
        WHERE parent_asin IN (SELECT * FROM unnest(?))
        GROUP BY scope_group
        ORDER BY scope_group
        """,
        [
            demo_scope.seller_parent_asins,
            str(CANONICAL_REVIEWS_PATH),
            all_scope_asins,
        ],
    ).fetchdf()
finally:
    connection.close()

assert scope_review_summary["product_count"].sum() == len(all_scope_asins)
display(scope_review_summary)

## 6. Conclusion and next decision

The displayed full-population checks reconcile the catalog one-to-one with metadata, show zero duplicate `parent_asin` values, and verify that catalog review totals cannot be inflated by a many-to-many metadata join. The coverage and candidate-niche tables above are the authoritative current counts; every canonical review in this snapshot joins to product metadata.

This is large enough for a serious first niche, but the product examples reveal that the leaf also contains leave-ins, detanglers, children's products, bars, sets, and other subtypes. Before marking the niche `approved`, we should decide whether the MVP should analyze the full Amazon leaf or a narrower product subtype. The demonstration workspace was not saved because no real seller ownership or competitor choice has been supplied yet.

## Вывод и следующее решение

Показанные проверки полной совокупности сверяют каталог с метаданными один-к-одному, подтверждают отсутствие повторяющихся `parent_asin` и исключают завышение числа отзывов из-за соединения many-to-many. Таблицы покрытия и кандидатной ниши выше являются источником актуальных точных значений; каждый канонический отзыв в этом срезе соединяется с метаданными товара.

Объема достаточно для серьезной первой ниши, но примеры показывают, что лист включает несмываемые кондиционеры (leave-in), средства для облегчения расчесывания (detanglers), детские продукты, твердые форматы, наборы и другие подтипы. До статуса `approved` («утверждено») нужно решить, анализирует ли MVP весь Amazon-лист или более узкий тип товара. Демонстрационная рабочая область не сохранена, потому что реальные товары продавца и конкуренты пока не указаны.